# 第一合法分钟与固定等待一根bar的同状态价值

只读已验证Qlib Recorder的固定原价股数五会话标签；先冻结`plan.json`。2021/2022经济值不进入本卡。

In [1]:
import os,json,hashlib
from pathlib import Path
os.environ['MLFLOW_ALLOW_FILE_STORE']='true';os.environ['MLFLOW_DISABLE_AGENT_HINT']='1'
import numpy as np,pandas as pd,qlib
from qlib.data import D
from src.core.recorder_sources import load_verified_recorder_objects
root=Path.cwd();card=root/'research/stage1_v5_one_bar_wait_value_audit_20261030';plan=json.loads((card/'plan.json').read_text())
sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
src=root/plan['source_card'];assert sha(src/'summary.json')==plan['source_summary_sha256']
assert sha(src/'verification_manifest.json')==plan['source_manifest_sha256']
assert json.loads((src/'nas_binding.json').read_text())['evidence']['sha256']==plan['source_nas_archive_sha256']
assert sha(root/plan['provider_uri']/'qlib_store_manifest.json')==plan['provider_manifest_sha256']
qlib.init(provider_uri=str(root/plan['provider_uri']),region='us',kernels=1)
cal=pd.DatetimeIndex(D.calendar(start_time='2018-10-15 09:31:00',end_time='2020-12-31 16:00:00',freq='1min'))
objects,_=load_verified_recorder_objects(src,expected_recorder_id=plan['source_recorder_id'],artifacts=['analysis/labels'],expected_manifest_sha256=plan['source_manifest_sha256'],expected_artifact_sha256={'analysis/labels':plan['source_artifact_sha256']})
full=objects['analysis/labels'];assert len(full)==385932
labels=full.loc[full.trade_date.dt.year.isin(plan['economic_years'])].copy()
assert labels.trade_date.dt.year.max()==2020 and len(labels)>0
assert labels.duplicated(['trade_date','bar_end_at']).sum()==0
assert pd.DatetimeIndex(labels.bar_end_at.unique()).isin(cal).all()
labels=labels.sort_values(['trade_date','bar_end_at']).reset_index(drop=True)
first=labels.groupby('trade_date',sort=True).nth(0).reset_index().sort_values('trade_date').reset_index(drop=True)
second=labels.groupby('trade_date',sort=True).nth(1).reset_index().sort_values('trade_date').reset_index(drop=True)
assert first.trade_date.equals(second.trade_date)
source_days=int(first.trade_date.nunique())
status_allowed=['paired_funded','attempt_no_fill']
source_ok=first.status.isin(status_allowed)&second.status.isin(status_allowed)
clock_ok=(second.bar_end_at-first.bar_end_at).eq(pd.Timedelta(minutes=1))
phase_ok=first.phase.eq(second.phase)
nav_ok=first.decision_nav.gt(0)&second.decision_nav.gt(0)
hold_error=(first.hold_terminal_nav-second.hold_terminal_nav).abs()
hold_ok=hold_error.le(1e-7)
valid=source_ok&clock_ok&phase_ok&nav_ok&hold_ok
assert not (source_ok&clock_ok&phase_ok&nav_ok&~hold_ok).any(), 'HOLD parity failed'
pairs=pd.DataFrame({'trade_date':first.trade_date,'year':first.trade_date.dt.year,
    'now_at':first.bar_end_at,'wait_at':second.bar_end_at,
    'now_status':first.status,'wait_status':second.status,
    'valid':valid,'hold_error_usd':hold_error})
for asset in ('soxx','soxl'):
    now=first[asset+'_value']*first.decision_nav
    wait=second[asset+'_value']*second.decision_nav
    pairs[asset+'_wait_minus_now']=(wait-now)/first.decision_nav
    pairs[asset+'_fee_wait_minus_now_usd']=second[asset+'_fee_usd']-first[asset+'_fee_usd']
    pairs[asset+'_share_wait_minus_now']=second[asset+'_filled_raw_shares']-first[asset+'_filled_raw_shares']
known=pairs.loc[pairs.valid].copy()
assert known[['soxx_wait_minus_now','soxl_wait_minus_now']].notna().all().all()
annual=[]
for year in plan['economic_years']:
    part=known.loc[known.year.eq(year)]
    annual.append({'year':int(year),'source_days':int(pairs.year.eq(year).sum()),'valid_first_pairs':int(len(part)),
      'soxx_mean_bp':float(part.soxx_wait_minus_now.mean()*1e4),'soxl_mean_bp':float(part.soxl_wait_minus_now.mean()*1e4),
      'soxx_positive_days':int(part.soxx_wait_minus_now.gt(0).sum()),'soxx_negative_days':int(part.soxx_wait_minus_now.lt(0).sum()),
      'soxx_fee_mean_usd':float(part.soxx_fee_wait_minus_now_usd.mean()),
      'soxx_share_mean':float(part.soxx_share_wait_minus_now.mean())})
for y,n in [(2018,40),(2019,200),(2020,200)]: assert next(a['valid_first_pairs'] for a in annual if a['year']==y)>=n
values=known.loc[known.year.isin([2019,2020]),'soxx_wait_minus_now'].to_numpy(float)
B=plan['information_gate']['bootstrap_replicates'];L=plan['information_gate']['block_sessions'];rng=np.random.default_rng(plan['information_gate']['seed'])
starts=rng.integers(0,len(values),size=(B,(len(values)+L-1)//L));idx=(starts[:,:,None]+np.arange(L)[None,None,:])%len(values);idx=idx.reshape(B,-1)[:,:len(values)]
lower=float(np.quantile(values[idx].mean(axis=1),plan['information_gate']['one_sided_lower_quantile']))
# Every causal adjacent intent is scanned; no pair supplies an online funded mask.
nxt=labels.groupby('trade_date',sort=False).shift(-1)
all_valid=labels.status.isin(status_allowed)&nxt.status.isin(status_allowed)&(nxt.bar_end_at-labels.bar_end_at).eq(pd.Timedelta(minutes=1))&labels.phase.eq(nxt.phase)&labels.decision_nav.gt(0)&nxt.decision_nav.gt(0)&(nxt.hold_terminal_nav-labels.hold_terminal_nav).abs().le(1e-7)
assert not (labels.status.isin(status_allowed)&nxt.status.isin(status_allowed)&(nxt.bar_end_at-labels.bar_end_at).eq(pd.Timedelta(minutes=1))&labels.phase.eq(nxt.phase)&labels.decision_nav.gt(0)&nxt.decision_nav.gt(0)&(nxt.hold_terminal_nav-labels.hold_terminal_nav).abs().gt(1e-7)).any()
all_diff=(nxt.soxx_value*nxt.decision_nav-labels.soxx_value*labels.decision_nav)/labels.decision_nav
all_diff=all_diff.loc[all_valid]
quarter=[]
for q,g in known.groupby(known.trade_date.dt.to_period('Q'),sort=True):
    quarter.append({'quarter':str(q),'days':int(len(g)),'soxx_mean_bp':float(g.soxx_wait_minus_now.mean()*1e4),'soxl_mean_bp':float(g.soxl_wait_minus_now.mean()*1e4)})
support_pass=all(next(a['valid_first_pairs'] for a in annual if a['year']==y)>=n for y,n in [(2018,40),(2019,200),(2020,200)])
value_pass=all(next(a['soxx_mean_bp'] for a in annual if a['year']==y)>0 for y in [2019,2020]) and lower>0
result={'experiment':plan['experiment'],'plan_sha256':sha(card/'plan.json'),'source_recorder_id':plan['source_recorder_id'],
 'source_days':source_days,'source_status_first_unknown':int((~source_ok).sum()),'source_clock_invalid':int((~clock_ok).sum()),
 'source_phase_mismatch':int((~phase_ok).sum()),'valid_first_pairs':int(len(known)),
 'hold_terminal_nav_max_error_usd':float(known.hold_error_usd.max()),'annual':annual,'quarter':quarter,
 'all_adjacent_valid_pairs':int(len(all_diff)),'all_adjacent_soxx_wait_minus_now_mean_bp':float(all_diff.mean()*1e4),
 'all_adjacent_soxx_positive_fraction':float(all_diff.gt(0).mean()),
 'soxx_first_pair_2019_2020_block_lower_bp':lower*1e4,
 'support_gate_passed':bool(support_pass),'fixed_wait_value_gate_passed':bool(value_pass),
 'new_model_fits':0,'new_accounts':0,'new_recorders':0,'historical_research_passed':False,
 'decision':'fixed_one_bar_wait_source_value_pass_information_unproved' if value_pass else 'fixed_one_bar_wait_value_failed_no_information_model'}
(card/'audit.json').write_text(json.dumps(result,ensure_ascii=False,indent=2,allow_nan=False)+'\n')
print(json.dumps(result,ensure_ascii=False,indent=2,allow_nan=False))


[60099:MainThread](2026-10-30 18:30:13,354) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[60099:MainThread](2026-10-30 18:30:13,939) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[60099:MainThread](2026-10-30 18:30:13,939) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/stage1_event_guard_fullminute_20261028')}


{
  "experiment": "stage1_v5_one_bar_wait_value_audit_20261030",
  "plan_sha256": "a9044953a6343fca451886bff99ee5890d645a3b36eaac754fb85919c133ec57",
  "source_recorder_id": "b85e80ab42fa4d9c83c6c9667114fd5d",
  "source_days": 558,
  "source_status_first_unknown": 4,
  "source_clock_invalid": 0,
  "source_phase_mismatch": 0,
  "valid_first_pairs": 554,
  "hold_terminal_nav_max_error_usd": 0.0,
  "annual": [
    {
      "year": 2018,
      "source_days": 53,
      "valid_first_pairs": 53,
      "soxx_mean_bp": 0.5216113330389303,
      "soxl_mean_bp": 0.18227603823884478,
      "soxx_positive_days": 37,
      "soxx_negative_days": 15,
      "soxx_fee_mean_usd": -0.00012086792452830441,
      "soxx_share_mean": -0.03773584905660377
    },
    {
      "year": 2019,
      "source_days": 252,
      "valid_first_pairs": 252,
      "soxx_mean_bp": -0.002084287895701917,
      "soxl_mean_bp": 0.08031168749469442,
      "soxx_positive_days": 118,
      "soxx_negative_days": 126,
      "soxx_fee

/var/folders/4n/bc8v7c391nq228754jxg84_w0000gn/T/ipykernel_60099/1131200458.py:28: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  clock_ok=(second.bar_end_at-first.bar_end_at).eq(pd.Timedelta(minutes=1))
/var/folders/4n/bc8v7c391nq228754jxg84_w0000gn/T/ipykernel_60099/1131200458.py:62: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  all_valid=labels.status.isin(status_allowed)&nxt.status.isin(status_allowed)&(nxt.bar_end_at-labels.bar_end_at).eq(pd.Timedelta(minutes=1))&labels.phase.eq(nxt.phase)&labels.decision_nav.gt(0)&nxt.decision_nav.gt(0)&(nxt.hold_terminal_nav-labels.hold_terminal_nav).abs().le(1e-7)
/var/folders/4n/bc8v7c391nq228754jxg84_w0000gn/T

# 固定等待一根分钟 bar 的同状态局部价值审计

**SOXX 判定：历史 Goal 未通过，本固定等待版关闭。**本卡只读已获 NAS 收据和限定读回的 Qlib 固定原价股数五会话标签 Recorder `b85e80ab42fa4d9c83c6c9667114fd5d`；没有新拟合、候选自身全账户或0/2/5bp SOXX 资金门。事前动作是在每天第一根合法完成分钟立即下一个资产的固定整股买单，或等待**恰好下一根合法分钟**再从当时父本自身状态计算同类买单。未来成交状态只用于标签审计，不作为在线筛选。

发现期2018–2020共558个来源日，554日形成相邻一分钟、同阶段、五会话终点一致的第一对；四个2020年来源路径未知日保留，HOLD终端NAV最大差0美元，来源支持门通过。SOXX等待−立即的同状态费后局部均值在部分2018Q4为+0.522bp、完整2019为**−0.002bp**、完整2020为**−0.022bp**；2019–2020 10会话区块单侧2.5%下界**−0.126bp**，事前两完整年均正与下界为正的价值门均失败。SOXL对应均值+0.182/+0.080/−0.108bp，也不支持跨年稳定收益。所有成熟同阶段相邻分钟共200950对，SOXX平均等待差−0.00063bp、正差比例45.09%；这些重叠对不能当独立样本。

区间原因：2019Q1 SOXX等待差−0.115bp、2019Q2 +0.045bp；2020Q1 −0.309bp、2020Q3 +0.251bp。完整2019/2020的负值不是手续费增加造成：两年等待减立即的SOXX平均手续费差分别约−0.000038/+0.000039美元，平均成交股数差−0.012/+0.012股。剩余价值同时包含不同入场价、整股数量和后续持仓价格路径，保存的两分支未单独识别各项因果份额；季度为事后归因，不能选择交易开关。

这是对“首分钟总买入是否只需推迟一根”的必要经济检验，并非所有动态等待策略或所有分钟信号无效的证明。固定一根等待在匹配发现期无稳定正价值，因此不基于它训练择时模型、不改等待长度或挑2018/Q3救援。2021/22经济值未查看；当前连续配置目标、完整 SOXX 账户及历史首次分钟送达/券商真实成交仍未资格。下一步仍需不同且当分钟可见的现金风险或资产选择信息机制，先证实分数到整股行动和同状态费用后价值，再考虑连续模型及完整账户。数值和冻结规则见`audit.json`、`plan.json`。